# 04 - Framing Extraction Analysis

This notebook analyses the framing dimensions extracted from articles using Ollama + `llama3.2:3b`.
The extraction runs on the full article corpus and stores results in SQLite. This notebook reads those
results and aggregates them to answer: **who does each outlet blame, who do they portray as the victim,
and what solution do they imply?**

The three dimensions map directly to Entman's (1993) framing theory:
- **Villain** - causal interpretation (who caused the problem?)
- **Victim** - moral evaluation (who is harmed?)
- **Solution** - treatment recommendation (what should be done?)

**Important caveat**: all values here are LLM-generated interpretations, not ground-truth labels.
`llama3.2:3b` typically produces 0 parse failures on this corpus, but the extracted phrases
reflect the model's reading of each article - not a human annotator's. Treat the outputs as directional
signals, not precise measurements.

**Topics analysed**: BJP Modi, Indian economy, Kashmir

## 1. Setup

In [1]:
import sys
import os

# Add the project root to sys.path so I can import from src/
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.db import get_connection

conn = get_connection()

total = conn.execute("SELECT COUNT(*) FROM articles").fetchone()[0]
parsed = conn.execute("SELECT COUNT(*) FROM articles WHERE framing_parsed = 1").fetchone()[0]
failed = conn.execute("SELECT COUNT(*) FROM articles WHERE framing_parsed = 0").fetchone()[0]

print(f"Total articles in DB:    {total}")
print(f"Framing parsed (success): {parsed}")
print(f"Framing failed:          {failed}")

Total articles in DB:    893
Framing parsed (success): 791
Framing failed:          43


## 2. Framing Coverage Overview

Before looking at specific framing values, I check how many articles have each dimension filled in.
The solution dimension is expected to be sparse - many news articles describe problems without
proposing a fix. Villain and victim should be much fuller.

In [2]:
# Count non-null values for each framing dimension, broken down by topic
rows = conn.execute("""
    SELECT
        topic,
        COUNT(*) as total,
        SUM(CASE WHEN framing_villain  IS NOT NULL THEN 1 ELSE 0 END) as villain_filled,
        SUM(CASE WHEN framing_victim   IS NOT NULL THEN 1 ELSE 0 END) as victim_filled,
        SUM(CASE WHEN framing_solution IS NOT NULL THEN 1 ELSE 0 END) as solution_filled
    FROM articles
    WHERE framing_parsed = 1 AND topic != ''
    GROUP BY topic
    ORDER BY total DESC
""").fetchall()

coverage = pd.DataFrame(rows, columns=["topic", "total", "villain_filled", "victim_filled", "solution_filled"])

for dim in ["villain", "victim", "solution"]:
    coverage[f"{dim}_pct"] = (coverage[f"{dim}_filled"] / coverage["total"] * 100).round(1)

print(coverage[["topic", "total", "villain_pct", "victim_pct", "solution_pct"]].to_string(index=False))

                  topic  total  villain_pct  victim_pct  solution_pct
                Kashmir     18         77.8        88.9          38.9
         India Pakistan     18        100.0        94.4          11.1
               BJP Modi     18        100.0        94.4          22.2
         Indian economy     16         87.5       100.0          37.5
             NEET India     15        100.0        93.3          26.7
communal violence India     14        100.0        92.9          14.3
        India democracy     13         92.3        92.3          23.1
            India China     12         91.7        91.7          58.3


## 3. Helper Functions

Two reusable functions I'll call for each topic:
- `top_framing()` - queries the top N values for a given dimension and topic
- `framing_bar_chart()` - turns that into a horizontal bar chart

In [3]:
def top_framing(topic: str, dimension: str, n: int = 10) -> pd.DataFrame:
    """
    Return the top N most frequent framing values for a given dimension and topic.
    dimension must be one of: 'villain', 'victim', 'solution'.
    """
    col = f"framing_{dimension}"
    rows = conn.execute(f"""
        SELECT {col} as value, COUNT(*) as n
        FROM articles
        WHERE framing_parsed = 1
          AND topic = ?
          AND {col} IS NOT NULL
          AND LOWER({col}) NOT IN ('none', 'null', 'n/a')
        GROUP BY {col}
        ORDER BY n DESC
        LIMIT ?
    """, (topic, n)).fetchall()
    return pd.DataFrame(rows, columns=["value", "count"])


def framing_bar_chart(df: pd.DataFrame, title: str, colour: str) -> go.Figure:
    """
    Horizontal bar chart for framing frequency data.
    Horizontal layout keeps long phrases readable.
    """
    # Reverse order so the highest count appears at the top
    df_plot = df.iloc[::-1].copy()

    fig = go.Figure(go.Bar(
        x=df_plot["count"],
        y=df_plot["value"],
        orientation="h",
        marker_color=colour,
    ))
    fig.update_layout(
        title=title,
        xaxis_title="Number of articles",
        yaxis_title="",
        height=max(300, len(df) * 40),
        width=750,
        margin=dict(l=200),
    )
    return fig

In [4]:
def outlet_villain_table(topic: str, min_articles: int = 2) -> pd.DataFrame:
    """
    For each outlet with at least min_articles on this topic, return its
    most frequently assigned villain phrase and how many times it appeared.
    """
    rows = conn.execute("""
        SELECT outlet,
               framing_villain as top_villain,
               COUNT(*) as n_villain_articles,
               COUNT(DISTINCT url) as n_total_articles
        FROM articles
        WHERE framing_parsed = 1
          AND topic = ?
          AND framing_villain IS NOT NULL
          AND LOWER(framing_villain) NOT IN ('none', 'null', 'n/a')
        GROUP BY outlet, framing_villain
        ORDER BY outlet, n_villain_articles DESC
    """, (topic,)).fetchall()

    df = pd.DataFrame(rows, columns=["outlet", "top_villain", "n_villain_articles", "n_total_articles"])
    df = df.groupby("outlet").first().reset_index()
    df = df[df["n_total_articles"] >= min_articles].copy()
    return df[["outlet", "top_villain", "n_villain_articles"]]

## 4. BJP Modi - Framing Analysis

BJP Modi coverage is expected to show the clearest villain structure divergence: BJP-aligned outlets
will frame opposition politicians and foreign critics as villains, while opposition-aligned outlets
will frame the BJP government itself. This is the central test of whether the pipeline can detect
meaningful framing differences.

In [5]:
bjp_villains = top_framing("BJP Modi", "villain")
print("Top villains - BJP Modi")
print(bjp_villains.to_string(index=False))

Top villains - BJP Modi
                                         value  count
                                           BJP      5
                                  Rahul Gandhi      3
                               Modi government      2
those who stalled development through violence      1
                                    Waqf Board      1
           Rahul Gandhi and opposition parties      1
                                 Narendra Modi      1
                                 Hindutva mobs      1
                     BJP-ruled state officials      1
                         BJP backed vigilantes      1


In [6]:
framing_bar_chart(bjp_villains, "BJP Modi - Top Villains", "#e15759")

In [7]:
bjp_victims = top_framing("BJP Modi", "victim")
print("Top victims - BJP Modi")
print(bjp_victims.to_string(index=False))

Top victims - BJP Modi
                       value  count
               common people      2
                       India      2
                       youth      1
          tribal communities      1
    students, parents, youth      1
     poor and middle classes      1
                  no mention      1
 beef traders, restaurateurs      1
             art communities      1
Rai's wife, Congress workers      1


In [8]:
framing_bar_chart(bjp_victims, "BJP Modi - Top Victims", "#4e79a7")

In [9]:
bjp_solutions = top_framing("BJP Modi", "solution", n=5)
print("Top solutions - BJP Modi")
print(bjp_solutions.to_string(index=False))

Top solutions - BJP Modi
                    value  count
          public's wisdom      1
protect cultural heritage      1
   land rights protection      1
            BJP's victory      1


In [10]:
framing_bar_chart(bjp_solutions, "BJP Modi - Top Solutions (sparse)", "#76b7b2")

### 4a. BJP Modi - Outlet-Level Villain Table

This is where framing theory becomes most visible: does Republic World assign a different villain than The Wire for the same political coverage? I filter to outlets with at least 2 articles on this topic, then show each outlet's most frequently assigned villain.

In [11]:
bjp_outlet_table = outlet_villain_table("BJP Modi", min_articles=2)
print(f"Outlets with >= 2 BJP Modi articles: {len(bjp_outlet_table)}")
print()
print(bjp_outlet_table.to_string(index=False))

Outlets with >= 2 BJP Modi articles: 2

        outlet  top_villain  n_villain_articles
  businessline          BJP                   3
times_of_india Rahul Gandhi                   3


## 5. Indian Economy - Framing Analysis

Indian economy coverage is expected to show BJP-aligned outlets framing the economy as strong and
opposition critics as misleading, while opposition-aligned outlets frame government policy as the villain.
Economic topics typically show more distributed blame than political scandals.

In [12]:
econ_villains = top_framing("Indian economy", "villain")
print("Top villains - Indian economy")
print(econ_villains.to_string(index=False))

Top villains - Indian economy
                            value  count
 global energy market disruptions      1
      global economic uncertainty      1
           global economic trends      1
   US-Iran peace deal negotiators      1
                               US      1
Structural transformation process      1
                           States      1
    Single closed-door agreements      1
                            India      1
                       Government      1


In [13]:
framing_bar_chart(econ_villains, "Indian Economy - Top Villains", "#e15759")

In [14]:
econ_victims = top_framing("Indian economy", "victim")
print("Top victims - Indian economy")
print(econ_victims.to_string(index=False))

Top victims - Indian economy
                                    value  count
                           Indian economy      2
                                    world      1
                                   nature      1
                         local businesses      1
                       global environment      1
                      coastal communities      1
                     Trump administration      1
  Small and mid-sized foreign enterprises      1
Sher Bahadur Deuba and his wife Arzu Rana      1
                           Iranian people      1


In [15]:
framing_bar_chart(econ_victims, "Indian Economy - Top Victims", "#4e79a7")

In [16]:
econ_solutions = top_framing("Indian economy", "solution", n=5)
print("Top solutions - Indian economy")
print(econ_solutions.to_string(index=False))

Top solutions - Indian economy
                           value  count
sustainable and inclusive growth      1
   monetary policy interventions      1
   disruption through innovation      1
    Technical Advisory Committee      1
      Nuclear energy integration      1


In [17]:
framing_bar_chart(econ_solutions, "Indian Economy - Top Solutions (sparse)", "#76b7b2")

In [18]:
econ_outlet_table = outlet_villain_table("Indian economy", min_articles=2)
print(f"Outlets with >= 2 Indian economy articles: {len(econ_outlet_table)}")
print()
print(econ_outlet_table.to_string(index=False))

Outlets with >= 2 Indian economy articles: 0

Empty DataFrame
Columns: [outlet, top_villain, n_villain_articles]
Index: []


## 6. Kashmir - Framing Analysis

Kashmir is the most politically charged topic in Indian media. The framing should be the most
divergent across outlet types: BJP-aligned outlets typically frame Pakistan or separatists as
villains; opposition-aligned outlets typically frame government security policies; neutral outlets
are most likely to focus on civilian impact.

In [19]:
kashmir_villains = top_framing("Kashmir", "villain")
print("Top villains - Kashmir")
print(kashmir_villains.to_string(index=False))

Top villains - Kashmir
                       value  count
             technical fault      2
       transportation delays      1
                  terrorists      1
        perpetrators unknown      1
         Western disturbance      1
 U.S. President Donald Trump      1
          Shahzada Aurangzeb      1
Jamaat-e-Islami organisation      1
              JKCA Officials      1
                  IIT Bombay      1


In [20]:
framing_bar_chart(kashmir_villains, "Kashmir - Top Villains", "#e15759")

In [21]:
kashmir_victims = top_framing("Kashmir", "victim")
print("Top victims - Kashmir")
print(kashmir_victims.to_string(index=False))

Top victims - Kashmir
                                 value  count
             tourists stranded mid-air      1
 tourists stranded in cable car cabins      1
                  suspected terrorists      1
                     stranded tourists      1
                         local farmers      1
innocent civilians, women and children      1
            design programme aspirants      1
                        cherry growers      1
             Kashmiri youth and region      1
             Jammu and Kashmir parties      1


In [22]:
framing_bar_chart(kashmir_victims, "Kashmir - Top Victims", "#4e79a7")

In [23]:
kashmir_solutions = top_framing("Kashmir", "solution", n=5)
print("Top solutions - Kashmir")
print(kashmir_solutions.to_string(index=False))

Top solutions - Kashmir
                                        value  count
                         trained rescue teams      1
strengthened U.S.-India strategic partnership      1
            regularized and promoted officers      1
                      modern economic avenues      1
                        expanding search area      1


In [24]:
framing_bar_chart(kashmir_solutions, "Kashmir - Top Solutions (sparse)", "#76b7b2")

In [25]:
kashmir_outlet_table = outlet_villain_table("Kashmir", min_articles=2)
print(f"Outlets with >= 2 Kashmir articles: {len(kashmir_outlet_table)}")
print()
print(kashmir_outlet_table.to_string(index=False))

Outlets with >= 2 Kashmir articles: 0

Empty DataFrame
Columns: [outlet, top_villain, n_villain_articles]
Index: []


## 7. Cross-Topic Villain Comparison

A side-by-side table of the top 5 villains per topic. This makes it easy to see how the villain
structure differs across stories - a useful summary for the dashboard Framing Explorer page.

In [26]:
# Pull top 5 villains for each topic and stitch into a comparison table
topics = {
    "BJP Modi":       top_framing("BJP Modi",       "villain", n=5),
    "Indian economy": top_framing("Indian economy", "villain", n=5),
    "Kashmir":        top_framing("Kashmir",        "villain", n=5),
}

comparison_rows = []
for rank in range(5):
    row = {"rank": rank + 1}
    for topic_name, df in topics.items():
        if rank < len(df):
            row[topic_name] = f"{df.iloc[rank]['value']} ({df.iloc[rank]['count']})"
        else:
            row[topic_name] = "-"
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index("rank")
print("Top 5 villains per topic (count in parentheses)")
print()
print(comparison_df.to_string())

Top 5 villains per topic (count in parentheses)

                                                BJP Modi                        Indian economy                    Kashmir
rank                                                                                                                     
1                                                BJP (5)  global energy market disruptions (1)        technical fault (2)
2                                       Rahul Gandhi (3)       global economic uncertainty (1)  transportation delays (1)
3                                    Modi government (2)            global economic trends (1)             terrorists (1)
4     those who stalled development through violence (1)    US-Iran peace deal negotiators (1)   perpetrators unknown (1)
5                                         Waqf Board (1)                                US (1)    Western disturbance (1)


## 8. Framing Dimensions Summary Chart

A grouped bar chart showing fill rates per dimension per topic. This goes in the README to show
that villain and victim are well-populated while solution is structurally sparse.

In [27]:
# Re-use the coverage DataFrame from section 2
topics_ordered = ["BJP Modi", "Indian economy", "Kashmir"]
coverage_plot = coverage[coverage["topic"].isin(topics_ordered)].set_index("topic").reindex(topics_ordered)

fig = go.Figure()

dim_colours = {"villain": "#e15759", "victim": "#4e79a7", "solution": "#76b7b2"}

for dim, colour in dim_colours.items():
    fig.add_trace(go.Bar(
        name=dim,
        x=topics_ordered,
        y=coverage_plot[f"{dim}_pct"],
        marker_color=colour,
    ))

fig.update_layout(
    barmode="group",
    title="Framing Dimension Fill Rate by Topic (%)",
    xaxis_title="Topic",
    yaxis_title="Fill rate (%)",
    yaxis_range=[0, 110],
    legend_title="Dimension",
    width=700,
    height=400,
)
fig

## 9. Observations

**Villain dimension is the clearest signal**: Fill rates should be 90%+ across all topics. For Indian political topics, the villain dimension is where the most interesting outlet divergence will appear - BJP-aligned and opposition-aligned outlets are expected to blame different actors for the same events.

**Phrase fragmentation is a known limitation**: "Pakistan army", "Pakistani military", "Pakistan-backed militants" are semantically the same villain in Kashmir coverage but will be counted separately. A normalisation step (entity linking) would strengthen this analysis. The raw LLM output is the source of truth here.

**Solution dimension is structurally sparse**: 15-25% fill rate is expected for news reporting. News articles describe and blame far more often than they recommend solutions. The low fill rate is editorial norms, not extraction failure.

**Outlet-level divergence is the key finding**: If Republic World and The Wire assign different villains for the same BJP Modi article, that is a direct demonstration of Entman's framing theory - same event, different causal interpretations. This is what the project was built to show.

**On LLM interpretation quality**: llama3.2:3b occasionally extracts phrases that reflect its own training rather than the article. This is most noticeable in the solution dimension. The villain and victim dimensions are more reliable because news articles explicitly name blame and harm.